# Diachronic Text Analysis — Dataset Construction, CBOW Model, and Training

This notebook documents the part of the project I developed after preprocessing: **shared vocabulary construction, dataset creation, CBOW model definition, training logic, and HPC execution**.

Compared with the previous version of this notebook, the code shown here is intentionally **shorter and more focused**: only the **essential excerpts** are reported, while the executable cells let me **demonstrate the real pipeline live** in front of the examiner.

## Relationship with the previous notebook

The notebook `diachronic_analysis.ipynb` covers the first half of the pipeline: data acquisition, cleaning, decade-level preprocessing and the construction of the shared vocabulary.

This notebook starts from the **shared vocabulary** and explains how the project moves from text to trainable embeddings:

**processed decade files → shared vocabulary → CBOW dataset → CBOW model → loss/optimizer → decade-specific checkpoints**

In [19]:
from pathlib import Path
import zipfile
import sys
import os

PROJECT_ROOT = Path("/home/vbucciero/diachronic_text_analysis")

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project folder not found and zip file unavailable.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
for rel in [
    "src/config.py",
    "src/data/dataset.py",
    "src/models/model.py",
    "src/models/train.py",
    "run_training.sbatch",
]:
    p = PROJECT_ROOT / rel
    print(f"{rel:<30} exists={p.exists()}")

Project root: /home/vbucciero/diachronic_text_analysis
src/config.py                  exists=True
src/data/dataset.py            exists=True
src/models/model.py            exists=True
src/models/train.py            exists=True
run_training.sbatch            exists=True


## 1. Central configuration

The second half of the pipeline is coordinated by `src/config.py`.  
This is important because the same file defines:

- the **decades**
- the **vocabulary size**
- the **context window**
- the **embedding dimension**
- the **training hyperparameters**
- the **input and output paths**

That centralization is essential in a diachronic project, because all decades must be trained under the **same structural assumptions**.

### Essential configuration excerpt (`src/config.py`)

```python
START_YEAR = 1900
END_YEAR = 2019
DECADES = [f"{year}s" for year in range(START_YEAR, END_YEAR + 1, 10)]

NGRAM_TYPE = 5
VOCAB_SIZE = 50001
UNK_TOKEN = "<UNK>"

EMBEDDING_DIM = 300
CONTEXT_WINDOW = 2 if NGRAM_TYPE == 5 else 1

BATCH_SIZE = 512
LEARNING_RATE = 0.025
EPOCHS = 3
```

This excerpt already shows the key design choices:
a **5-gram setting**, a **shared 50k vocabulary**, **300-dimensional embeddings**, and a **window of 2 words on each side**.

In [11]:
from src.config import (
    NGRAM_TYPE, CONTEXT_WINDOW, VOCAB_SIZE, UNK_TOKEN,
    EMBEDDING_DIM, BATCH_SIZE, LEARNING_RATE, EPOCHS,
    DECADES, get_vocab_path, get_model_path
)

print("Configuration summary")
print("-" * 60)
print("N-gram type:      ", NGRAM_TYPE)
print("Context window:   ", CONTEXT_WINDOW)
print("Vocabulary size:  ", VOCAB_SIZE)
print("UNK token:        ", UNK_TOKEN)
print("Embedding dim:    ", EMBEDDING_DIM)
print("Batch size:       ", BATCH_SIZE)
print("Learning rate:    ", LEARNING_RATE)
print("Epochs:           ", EPOCHS)
print("Number of decades:", len(DECADES))
print("Vocabulary path:  ", get_vocab_path())

Configuration summary
------------------------------------------------------------
N-gram type:       5
Context window:    2
Vocabulary size:   50001
UNK token:         <UNK>
Embedding dim:     300
Batch size:        512
Learning rate:     0.025
Epochs:            3
Number of decades: 12
Vocabulary path:   /home/vbucciero/diachronic_text_analysis/data/processed/5gram-expanded/vocab.json


## 2. Shared vocabulary construction

A diachronic model needs a **stable word-to-index mapping** across time.  
Without a shared vocabulary, the embedding spaces of different decades would not even refer to the same lexical inventory.

The implementation is in `src/data/build_vocab.py`.

### Essential vocabulary excerpt (`src/data/build_vocab.py`)

```python
def compute_total_frequencies(decade_files):
    total_freq = defaultdict(int)
    for file_path in decade_files:
        decade_freq = load_decade_frequencies(file_path)
        for word, freq in decade_freq.items():
            total_freq[word] += freq
    return dict(total_freq), decade_frequencies

def build_vocabulary(total_freq):
    num_words_to_select = VOCAB_SIZE - 1
    sorted_words = sorted(total_freq.items(), key=lambda x: x[1], reverse=True)
    selected = sorted_words[:num_words_to_select]

    vocab = {UNK_TOKEN: 0}
    for idx, (word, _) in enumerate(selected, start=1):
        vocab[word] = idx
    return vocab
```

The core idea is simple and methodologically correct:

1. aggregate frequencies over **all decades**
2. select the most frequent words
3. reserve index **0** for `<UNK>`
4. save a single `vocab.json` used by every dataset and every model

In [20]:
from pathlib import Path
from src.data.build_vocab import get_decade_files, compute_total_frequencies, build_vocabulary
from src.config import get_preprocess_output_dir

processed_dir = get_preprocess_output_dir(expanded=False)
decade_files = get_decade_files(processed_dir) if Path(processed_dir).exists() else []

print("Processed directory:", processed_dir)
print("Decade files found:", len(decade_files))

if decade_files:
    total_freq, _ = compute_total_frequencies(decade_files[:min(2, len(decade_files))])
    vocab_preview = build_vocabulary(total_freq)
    print("Preview vocab size:", len(vocab_preview))
    print("First 10 entries:", list(vocab_preview.items())[:10])
else:
    print("No preprocessed decade files are bundled here, so this cell demonstrates the real functions without rebuilding the full vocabulary.")

Processed directory: /home/vbucciero/diachronic_text_analysis/data/processed/5gram-expanded
Decade files found: 12
  Loading frequencies from all decades...
    1900s... 79,734 words
    1910s... 73,427 words

  ✓ Total unique words: 91,110

  Building vocabulary (top 50,000 words + 1 UNK)...
  ✓ Created vocabulary: 50,001 total words (including <UNK>)
Preview vocab size: 50001
First 10 entries: [('<UNK>', 0), ('the', 1), ('of', 2), ('that', 3), ('on', 4), ('to', 5), ('and', 6), ('his', 7), ('is', 8), ('in', 9)]


## 3. Dataset creation

The project transforms each processed decade file into **CBOW supervision pairs**.  
This is implemented in `src/data/dataset.py`.

The file contains multiple historical versions, but the important point is that the **final class definition is the active one used by the trainer**.

### Essential dataset excerpt (`src/data/dataset.py`)

```python
class CBOWDataset(Dataset):
    def __init__(self, decade: str):
        self.decade = decade
        self._load_vocab()
        self.contexts, self.targets = self._load_data_optimized()

    def _load_vocab(self):
        with open(VOCAB_FILE, "r", encoding="utf-8") as f:
            self.word2idx = json.load(f)
        self.unk_idx = self.word2idx.get(UNK_TOKEN, 0)

    def __getitem__(self, idx):
        return self.targets[idx], self.contexts[idx]
```

This is the essential behavior:

- load the **shared vocabulary**
- convert n-grams into **integer ids**
- generate `(target, context)` examples
- store them in a compact tensor-based representation

### Sliding-window logic used by the dataset

For each token position, the model predicts the target word from the surrounding context.

For a 5-gram `[A, B, C, D, E]` and `window = 2`, the dataset generates examples such as:

- target `A` → context `[PAD, PAD, B, C]`
- target `B` → context `[PAD, A, C, D]`
- target `C` → context `[A, B, D, E]`
- target `D` → context `[B, C, E, PAD]`
- target `E` → context `[C, D, PAD, PAD]`

So one n-gram yields **multiple supervised examples**, not just one.

In [21]:
# Runnable toy demonstration of the same sliding-window idea

tokens = ["A", "B", "C", "D", "E"]
window = 2
PAD = "<PAD>"

examples = []
for i, target in enumerate(tokens):
    left = [tokens[i-w] if i-w >= 0 else PAD for w in range(window, 0, -1)]
    right = [tokens[i+w] if i+w < len(tokens) else PAD for w in range(1, window + 1)]
    context = left + right
    examples.append((target, context))

for target, context in examples:
    print(f"target={target:>3} | context={context}")

target=  A | context=['<PAD>', '<PAD>', 'B', 'C']
target=  B | context=['<PAD>', 'A', 'C', 'D']
target=  C | context=['A', 'B', 'D', 'E']
target=  D | context=['B', 'C', 'E', '<PAD>']
target=  E | context=['C', 'D', '<PAD>', '<PAD>']


### Error handling in dataset creation

The dataset code explicitly checks for missing resources.

```python
if not os.path.exists(VOCAB_FILE):
    raise FileNotFoundError(f"Vocabolario non trovato: {VOCAB_FILE}")

if not os.path.exists(decade_file):
    raise FileNotFoundError(f"Dati non trovati: {decade_file}")
```

This is important in practice, because training can only start if:

- `vocab.json` exists
- the processed file for the requested decade exists

In [14]:
# Safe live check of the resources expected by the dataset

from src.config import VOCAB_FILE, get_processed_file_path

print("Vocabulary file exists:", Path(VOCAB_FILE).exists())
for decade in DECADES[:3]:
    print(f"{decade:<6} processed file exists:", Path(get_processed_file_path(decade)).exists())

Vocabulary file exists: True
1900s  processed file exists: True
1910s  processed file exists: True
1920s  processed file exists: True


## 4. CBOW model definition

The neural architecture is implemented in `src/models/model.py`.  
The model is intentionally simple and appropriate for distributional semantics: it embeds the context, averages it, and predicts the target word.

### Essential model excerpt (`src/models/model.py`)

```python
class CBOWModel(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM):
        super().__init__()
        self.embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, context_words):
        embedded = self.embeddings(context_words)
        pooled = embedded.mean(dim=1)
        logits = self.output_layer(pooled)
        return logits
```

This is the whole CBOW computation:

1. **embedding lookup**
2. **average pooling**
3. **linear projection to vocabulary logits**

In [ ]:
import torch
from src.models.model import CBOWModel
from src.config import VOCAB_SIZE, EMBEDDING_DIM, CONTEXT_WINDOW

model = CBOWModel(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM)
dummy_contexts = torch.randint(0, 20, (4, 2 * CONTEXT_WINDOW))
logits = model(dummy_contexts)

print("Dummy input shape:   ", tuple(dummy_contexts.shape))
print("Logits shape:        ", tuple(logits.shape))
print("Embedding matrix:    ", tuple(model.embeddings.weight.shape))
print("Output layer matrix: ", tuple(model.output_layer.weight.shape))

✓ Modello CBOW creato:
  - Vocabolario: 50,001 token
  - Dimensione embedding: 300
  - Parametri totali: 30,050,601
Dummy input shape:    (4, 4)
Logits shape:         (4, 50001)
Embedding matrix:     (50001, 300)
Output layer matrix:  (50001, 300)


### `<UNK>` and padding inside the model

The project uses index `0` as the special fallback index in the embedding layer through `padding_idx=0`.

Conceptually, the two roles are different:

- **`<UNK>`** means *a real word that is outside the selected vocabulary*
- **padding** means *an empty position inserted only to keep the context length fixed*

In the implementation, the important practical fact is that index `0` is treated as a **special non-informative index** in the embedding layer, so boundary padding does not update gradients like ordinary lexical items.

## 5. Loss function, optimizer, and training loop

The training logic is implemented in `src/models/train.py`.  
Again, here I keep only the essential part of the code.

### Essential training excerpt (`src/models/train.py`)

```python
dataset = CBOWDataset(decade=decade)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = CBOWModel(len(dataset.word2idx), EMBEDDING_DIM).to(self.device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    for targets, contexts in dataloader:
        optimizer.zero_grad()
        outputs = model(contexts)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
```

This is the actual learning mechanism:

- the dataset produces `(target, context)` pairs
- the model predicts the target distribution
- `CrossEntropyLoss` measures the prediction error
- `Adam` updates the parameters

### Error handling in training

The trainer also contains explicit defensive checks:

```python
try:
    dataset = CBOWDataset(decade=decade)
except Exception as e:
    print(f"❌ Errore dataset: {e}")
    return None

if len(dataset) == 0:
    print("⚠️ Dataset vuoto o window troppo grande per i dati. Salto.")
    return None
```

This is useful to explain at the exam because it shows that the pipeline was designed not only to work in the happy path, but also to **fail clearly** when resources are missing or unusable.

In [16]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LinearLR
from tqdm import tqdm
import json
import glob

# Importiamo config. Se SHOW_PROGRESS_BAR non c'è, mettiamo True di default
try:
    from src.config import SHOW_PROGRESS_BAR
except ImportError:
    SHOW_PROGRESS_BAR = True

from src.config import (
    DECADES, BATCH_SIZE, EPOCHS, LEARNING_RATE,
    EMBEDDING_DIM, VOCAB_SIZE, MODELS_DIR, get_model_path,
)
from src.data.dataset import CBOWDataset
from src.models.model import CBOWModel

class Trainer:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.scaler = torch.amp.GradScaler('cuda')
        print(f"🚀 Trainer inizializzato su {self.device}")

    def train_decade(self, decade: str):
        print(f"\n{'='*60}\n📚 Training: {decade}\n{'='*60}")

        # 1. DATASET
        try:
            dataset = CBOWDataset(decade=decade)
        except Exception as e:
            print(f"❌ Errore dataset: {e}")
            return None
            
        if len(dataset) == 0:
            print("⚠️ Dataset vuoto o window troppo grande per i dati. Salto.")
            return None

        dataloader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4 if self.device == "cuda" else 0,
            pin_memory=True
        )

        # 2. MODELLO
        actual_vocab_size = len(dataset.word2idx)
        model = CBOWModel(actual_vocab_size, EMBEDDING_DIM).to(self.device)
        criterion = nn.CrossEntropyLoss()
        
        # OTTIMIZZATORE & SCHEDULER (Novità!)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        
        # Scheduler Lineare: Parte da LR pieno e scende fino all'1% alla fine delle epoche
        # Questo stabilizza enormemente i vettori finali.
        scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.01, total_iters=EPOCHS)

        # 3. LOOP
        model.train()
        for epoch in range(EPOCHS):
            total_loss = 0
            
            # Gestione barra progresso intelligente (non intasa i log se disabilitata)
            pbar = tqdm(dataloader, desc=f"Ep {epoch+1}/{EPOCHS}", disable=not SHOW_PROGRESS_BAR)
            
            for targets, contexts in pbar:
                targets, contexts = targets.to(self.device), contexts.to(self.device)

                optimizer.zero_grad()
                
                # Mixed Precision
                with torch.cuda.amp.autocast(enabled=(self.device=="cuda")):
                    outputs = model(contexts)
                    loss = criterion(outputs, targets)

                self.scaler.scale(loss).backward()
                self.scaler.step(optimizer)
                self.scaler.update()

                total_loss += loss.item()
                if SHOW_PROGRESS_BAR:
                    pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.6f}"})

            # Step dello Scheduler alla fine dell'epoca
            scheduler.step()
            
            avg_loss = total_loss / len(dataloader)
            current_lr = scheduler.get_last_lr()[0]
            print(f"  Epoch {epoch+1} completata. Avg Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

            # CHECKPOINTING (Salvataggio intermedio)
            # Salva una copia del modello ad ogni epoca. Utile se crasha o per analisi.
            epoch_path = os.path.join(MODELS_DIR, f"emb_{decade}_ep{epoch+1}.pt")
            torch.save(model.state_dict(), epoch_path)

        # 4. SALVA IL MODELLO FINALE (Sovrascrive o crea quello "pulito")
        print(f"💾 Salvataggio modello finale...")
        final_path = get_model_path(decade)
        torch.save(model.state_dict(), final_path)
        
        # Pulizia opzionale: rimuovi i checkpoint intermedi per risparmiare spazio?
        # Per ora li lasciamo, meglio averli.
        
        return {"decade": decade, "final_loss": avg_loss, "training_time": "N/A"}

    def train_all_decades(self):
        metrics = {}
        for decade in DECADES:
            res = self.train_decade(decade)
            if res: metrics[decade] = res
            
        # Salva sommario
        with open(os.path.join(MODELS_DIR, "training_summary.json"), 'w') as f:
            json.dump(metrics, f, indent=2)

def main():
    os.makedirs(MODELS_DIR, exist_ok=True)
    trainer = Trainer()
    trainer.train_all_decades()

if __name__ == "__main__":
    sys.exit(main())

🚀 Trainer inizializzato su cuda

📚 Training: 1900s
  - Caricamento dati grezzi da /home/vbucciero/diachronic_text_analysis/data/processed/5gram-expanded/1900s.txt...
    Righe mescolate. Generazione Tensori...
    Conversione in Tensor Unici (questo libera la RAM)...
✓ Dataset '1900s' pronto (Memory Optimized).
  - Esempi totali: 80,120,390
  - RAM stimata Tensori: 3056.35 MB
✓ Modello CBOW creato:
  - Vocabolario: 50,002 token
  - Dimensione embedding: 300
  - Parametri totali: 30,051,202


Ep 1/3:   0%|          | 0/156486 [00:00<?, ?it/s]/tmp/ipykernel_2180084/2131493975.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(self.device=="cuda")):
Ep 1/3:   1%|          | 1508/156486 [00:12<21:43, 118.88it/s, loss=5.5223, lr=0.025000] 


KeyboardInterrupt: 

In [22]:
# Live inspection of the checkpoints that training is expected to produce

from src.config import get_model_path

print("Expected checkpoint paths")
print("-" * 60)
for decade in DECADES:
    print(decade, "->", get_model_path(decade))

Expected checkpoint paths
------------------------------------------------------------
1900s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1900s.pt
1910s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1910s.pt
1920s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1920s.pt
1930s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1930s.pt
1940s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1940s.pt
1950s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1950s.pt
1960s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1960s.pt
1970s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1970s.pt
1980s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1980s.pt
1990s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_1990s.pt
2000s -> /home/vbucciero/diachronic_text_analysis/models/5gram-full/emb_2000s.pt
2010s -> /home/vbuccie

In [30]:
import json

with open("/home/vbucciero/diachronic_text_analysis/models/5gram-full/training_summary.json") as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "1900s": {
    "decade": "1900s",
    "final_loss": 4.140800550209986,
    "training_time": "N/A"
  },
  "1910s": {
    "decade": "1910s",
    "final_loss": 4.0224784581747315,
    "training_time": "N/A"
  },
  "1920s": {
    "decade": "1920s",
    "final_loss": 3.9953917964724717,
    "training_time": "N/A"
  },
  "1930s": {
    "decade": "1930s",
    "final_loss": 3.9037446366428004,
    "training_time": "N/A"
  },
  "1940s": {
    "decade": "1940s",
    "final_loss": 3.9063933375729007,
    "training_time": "N/A"
  },
  "1950s": {
    "decade": "1950s",
    "final_loss": 4.059048911402821,
    "training_time": "N/A"
  },
  "1960s": {
    "decade": "1960s",
    "final_loss": 4.228177775370082,
    "training_time": "N/A"
  },
  "1970s": {
    "decade": "1970s",
    "final_loss": 4.283423981899469,
    "training_time": "N/A"
  },
  "1980s": {
    "decade": "1980s",
    "final_loss": 4.366048778753103,
    "training_time": "N/A"
  },
  "1990s": {
    "decade": "1990s",
    "final_lo

## 6. HPC execution with SLURM

The full training was designed to run on a cluster through `run_training.sbatch`.  
That matters because the project is not only a notebook experiment: it is a **reproducible training pipeline**.

### Essential SLURM excerpt (`run_training.sbatch`)

```bash
#SBATCH --job-name=diachronic_cbow_training
#SBATCH --output=logs/training_%j.log
#SBATCH --error=logs/training_%j.err
#SBATCH --time=24:00:00
#SBATCH --mem=32G
#SBATCH --cpus-per-task=8
#SBATCH --partition=gpu

module load python/3.11
module load cuda/12.2
source .venv/bin/activate

python -m src.models.train
```

This script connects the code to the actual execution environment:
requested resources, loaded CUDA modules, activated environment, and launched training.

In [9]:
# Quick preview of the real SBATCH file used in the repository

sbatch_path = PROJECT_ROOT / "run_training.sbatch"
with open(sbatch_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines[:20]:
    print(line.rstrip())

#!/bin/bash

#SBATCH --job-name=diachronic_cbow_training
#SBATCH --output=logs/training_%j.log
#SBATCH --error=logs/training_%j.err
#SBATCH --time=24:00:00
#SBATCH --mem=32G
#SBATCH --cpus-per-task=8
#SBATCH --partition=gpu

# ============================================================================
# SBATCH per Training CBOW (FASE 5)
# ============================================================================
# Esecuzione:
#   sbatch run_training.sbatch
#
# Monitoraggio:
#   squeue -u $USER
#   tail -f logs/training_*.log
#


## 7. What I should execute live during the exam

The most useful cells to run in front of the professor are:

1. **setup cell** to show that the notebook is connected to the real repository  
2. **configuration summary** to explain the global design choices  
3. **vocabulary preview** to show how the shared vocabulary is built  
4. **sliding-window demo** to show how one n-gram becomes CBOW examples  
5. **model sanity-check cell** to show input/output dimensions  
6. **mini training step** to demonstrate that loss computation and optimization really work

This gives a clean live narrative: from processed text, to dataset, to model, to training.

## 8. Final conclusion

This phase of the project is the bridge between **historical text** and **diachronic embeddings**.

Its technical value is that it creates a pipeline which is:

- **temporally consistent**, because all decades share the same vocabulary
- **computationally feasible**, because the dataset is tensor-based and training is batched
- **methodologically transparent**, because the CBOW architecture and the loss are simple and interpretable
- **ready for the next phase**, because the output is one checkpoint per decade for later alignment and semantic-drift analysis